In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

print(os.listdir('/content/drive/MyDrive'))

['certificate (1).jpg', 'certificate (2).jpg', 'certificate (3).jpg', 'certificate (4).jpg', 'certificate (6).jpg', 'certificate (5).jpg', 'certificate (9).jpg', 'certificate (7).jpg', 'certificate (8).jpg', 'certificate (10).jpg', 'certificate (11).jpg', 'certificate (13).jpg', 'certificate.jpg', 'certificate (12).jpg', 'Class 6 test', 'DSA Snake Game Output.mp4', 'IMG-20240929-WA0005.jpg', '1000058530.jpg', 'TOC & CD DA3 Video.mp4', 'Photography Club Leads – Timetable Submission Form  (File responses)', 'Photography Club Leads – Timetable Submission Form .gform', 'Photography Club Leads – Timetable Submission Form  (Responses).gsheet', 'Untitled form (File responses) (1)', 'Untitled form.gform', 'Carnival Stalls.gsheet', 'TechnoVIT Budget PC.gsheet', 'Photography Club (VIT Chennai) .gform', 'CodeChef EM Interview Slots (18 10 25).gsheet', 'Sheet.gsheet', 'ML Project', 'Untitled form (File responses)', 'CodeChef VITC Testimonials Leads.gform', 'HEATCODE OC.gsheet', 'Photography Club (

In [ ]:
import os

project_path = "/content/drive/MyDrive/AML_Project"

print("Contents of AML_Project:")
print(os.listdir(project_path))

Contents of AML_Project:
['notebooks', 'dataset', 'outputs', 'dashboard', 'report']


In [ ]:
import os

dataset_path = "/content/drive/MyDrive/AML_Project/dataset"

print("Files in dataset folder:")
print(os.listdir(dataset_path))

Files in dataset folder:
['cities.csv', 'customers.csv', 'atms.csv', 'merchants.csv', 'transactions.csv', 'fraud_transactions_v1.csv']


In [ ]:
import pandas as pd

transactions_path = "/content/drive/MyDrive/AML_Project/dataset/transactions.csv"

transactions = pd.read_csv(transactions_path)

print("Transactions loaded successfully!")
print("Shape:", transactions.shape)

transactions.head()

Transactions loaded successfully!
Shape: (148544, 14)


,transaction_id,sender,receiver,receiver_name,is_merchant,transaction_type,amount,timestamp,atm_id,city,state,latitude,longitude,fraud_type
0,TX00007570,CUST01729,MER00193,Bharat Petroleum,True,Fuel,3002,2026-01-01 07:00:03,ATM0172,Delhi,Delhi,28.574159,77.160227,Normal
1,TX00017239,CUST04035,MER00158,Bharat Petroleum,True,Fuel,620,2026-01-01 07:00:06,ATM0111,Patna,Bihar,25.642494,85.148727,Normal
2,TX00010111,CUST02337,CUST01403,CUST01403,False,Transfer,33077,2026-01-01 07:00:08,ATM0068,Bhopal,Madhya Pradesh,23.255729,77.462445,Normal
3,TX00009240,CUST02134,CUST02339,CUST02339,False,Card,6389,2026-01-01 07:00:10,ATM0171,Jaipur,Rajasthan,26.917736,75.807042,Normal
4,TX00009635,CUST02235,CUST04794,CUST04794,False,ATM,3950,2026-01-01 07:00:11,ATM0155,Chandigarh,Chandigarh,30.766373,76.738066,Normal


In [ ]:
print("Columns:")
print(transactions.columns.tolist())

Columns:
['transaction_id', 'sender', 'receiver', 'receiver_name', 'is_merchant', 'transaction_type', 'amount', 'timestamp', 'atm_id', 'city', 'state', 'latitude', 'longitude', 'fraud_type']


In [ ]:
print(transactions.shape)
print(transactions.columns.tolist())

(148544, 14)
['transaction_id', 'sender', 'receiver', 'receiver_name', 'is_merchant', 'transaction_type', 'amount', 'timestamp', 'atm_id', 'city', 'state', 'latitude', 'longitude', 'fraud_type']


In [ ]:

fraud_transactions = transactions.copy()

# Convert timestamp to datetime
fraud_transactions["timestamp"] = pd.to_datetime(
    fraud_transactions["timestamp"],
    errors="coerce"
)

# Reset transaction-level fraud labels
fraud_transactions["fraud_type"] = "Normal"

# Make sure amount is numeric
fraud_transactions["amount"] = pd.to_numeric(
    fraud_transactions["amount"],
    errors="coerce"
)

print("Fraud injection dataset prepared.")
print("Shape:", fraud_transactions.shape)

print("\nFraud labels:")
print(fraud_transactions["fraud_type"].value_counts())

print("\nTimestamp range:")
print(
    fraud_transactions["timestamp"].min(),
    "to",
    fraud_transactions["timestamp"].max()
)

Fraud injection dataset prepared.
Shape: (148544, 14)

Fraud labels:
fraud_type
Normal    148544
Name: count, dtype: int64

Timestamp range:
2026-01-01 07:00:03 to 2026-01-07 22:59:59


In [ ]:

existing_transaction_ids = set(
    fraud_transactions["transaction_id"].astype(str)
)

print("Existing transaction IDs:", len(existing_transaction_ids))

Existing transaction IDs: 148544


In [ ]:

fraud_counter = 1

def generate_fraud_id():
    global fraud_counter

    while True:
        tx_id = f"FRAUD{fraud_counter:06d}"
        fraud_counter += 1

        if tx_id not in existing_transaction_ids:
            existing_transaction_ids.add(tx_id)
            return tx_id

In [ ]:
import random
from datetime import timedelta

def create_layering_pattern(fraud_transactions, num_patterns=50):

    new_transactions = []

    # Get customer-to-customer transactions only
    customer_ids = list(
        set(fraud_transactions["sender"].astype(str)) |
        set(fraud_transactions["receiver"].astype(str))
    )

    # Remove merchant-style IDs if present
    customer_ids = [
        x for x in customer_ids
        if x.startswith("CUST")
    ]

    print("Available customer accounts:", len(customer_ids))

    for pattern_num in range(num_patterns):

        # Need at least 5 different accounts
        accounts = random.sample(customer_ids, 5)

        source = accounts[0]
        intermediary1 = accounts[1]
        intermediary2 = accounts[2]
        intermediary3 = accounts[3]
        final_account = accounts[4]

        # Starting time
        start_time = pd.Timestamp("2026-01-03") + pd.Timedelta(
            minutes=random.randint(0, 60 * 24 * 4)
        )

        # Suspicious amount
        amount = round(random.uniform(50000, 200000), 2)

        chain = [
            (source, intermediary1),
            (intermediary1, intermediary2),
            (intermediary2, intermediary3),
            (intermediary3, final_account)
        ]

        for step, (sender, receiver) in enumerate(chain):

            tx_time = start_time + pd.Timedelta(
                minutes=step * random.randint(2, 8)
            )

            row = {
                "transaction_id": generate_fraud_id(),
                "sender": sender,
                "receiver": receiver,
                "receiver_name": receiver,
                "is_merchant": False,
                "transaction_type": "Transfer",
                "amount": amount,
                "timestamp": tx_time,
                "atm_id": None,
                "city": "Delhi",
                "state": "Delhi",
                "latitude": 28.6139,
                "longitude": 77.2090,
                "fraud_type": "Layering"
            }

            new_transactions.append(row)

    new_df = pd.DataFrame(new_transactions)

    return pd.concat(
        [fraud_transactions, new_df],
        ignore_index=True
    )


# Inject layering
fraud_transactions = create_layering_pattern(
    fraud_transactions,
    num_patterns=50
)

print("Layering injection completed!")
print("New dataset shape:", fraud_transactions.shape)

print("\nFraud type distribution:")
print(fraud_transactions["fraud_type"].value_counts())

Available customer accounts: 5000
Layering injection completed!
New dataset shape: (148744, 14)

Fraud type distribution:
fraud_type
Normal      148544
Layering       200
Name: count, dtype: int64


In [ ]:

layering = fraud_transactions[
    fraud_transactions["fraud_type"] == "Layering"
].copy()

layering = layering.sort_values(
    ["sender", "timestamp"]
)

print("Layering transactions:", len(layering))
print("Unique layering senders:", layering["sender"].nunique())
print()

display(
    layering[
        [
            "transaction_id",
            "sender",
            "receiver",
            "transaction_type",
            "amount",
            "timestamp",
            "city",
            "fraud_type"
        ]
    ].head(20)
)

Layering transactions: 200
Unique layering senders: 198



,transaction_id,sender,receiver,transaction_type,amount,timestamp,city,fraud_type
148546,FRAUD000003,CUST00044,CUST03607,Transfer,162902.59,2026-01-06 12:00:00,Delhi,Layering
148601,FRAUD000058,CUST00046,CUST04966,Transfer,189740.43,2026-01-06 19:01:00,Delhi,Layering
148559,FRAUD000016,CUST00049,CUST02595,Transfer,123563.18,2026-01-03 20:34:00,Delhi,Layering
148557,FRAUD000014,CUST00075,CUST01530,Transfer,123563.18,2026-01-03 20:26:00,Delhi,Layering
148677,FRAUD000134,CUST00107,CUST02187,Transfer,152102.66,2026-01-03 22:33:00,Delhi,Layering
148637,FRAUD000094,CUST00107,CUST01277,Transfer,139982.26,2026-01-05 22:39:00,Delhi,Layering
148628,FRAUD000085,CUST00121,CUST00481,Transfer,109317.29,2026-01-03 06:10:00,Delhi,Layering
148632,FRAUD000089,CUST00183,CUST04521,Transfer,59906.96,2026-01-04 23:43:00,Delhi,Layering
148563,FRAUD000020,CUST00256,CUST04863,Transfer,180029.11,2026-01-04 20:21:00,Delhi,Layering
148709,FRAUD000166,CUST00261,CUST01956,Transfer,121670.78,2026-01-04 14:23:00,Delhi,Layering


In [ ]:
# Cell 10 — Recreate Layering Injection with Pattern IDs

import pandas as pd
import random

# --------------------------------------------------
# 1. Reset to original normal transactions
# --------------------------------------------------

fraud_transactions = transactions.copy()

fraud_transactions["timestamp"] = pd.to_datetime(
    fraud_transactions["timestamp"],
    errors="coerce"
)

fraud_transactions["amount"] = pd.to_numeric(
    fraud_transactions["amount"],
    errors="coerce"
)

fraud_transactions["fraud_type"] = "Normal"
fraud_transactions["pattern_id"] = None

# Existing transaction IDs
existing_transaction_ids = set(
    fraud_transactions["transaction_id"].astype(str)
)

fraud_counter = 1


# --------------------------------------------------
# 2. Unique transaction ID generator
# --------------------------------------------------

def generate_fraud_id():
    global fraud_counter

    while True:
        tx_id = f"FRAUD{fraud_counter:06d}"
        fraud_counter += 1

        if tx_id not in existing_transaction_ids:
            existing_transaction_ids.add(tx_id)
            return tx_id


# --------------------------------------------------
# 3. Get customer accounts
# --------------------------------------------------

customer_ids = sorted(
    set(fraud_transactions["sender"].astype(str))
    |
    set(fraud_transactions["receiver"].astype(str))
)

customer_ids = [
    x for x in customer_ids
    if x.startswith("CUST")
]

print("Available customer accounts:", len(customer_ids))


# --------------------------------------------------
# 4. Generate layering patterns
# --------------------------------------------------

new_transactions = []

NUM_PATTERNS = 50

for pattern_num in range(1, NUM_PATTERNS + 1):

    # Five accounts create a four-step chain
    accounts = random.sample(customer_ids, 5)

    source = accounts[0]
    intermediary1 = accounts[1]
    intermediary2 = accounts[2]
    intermediary3 = accounts[3]
    final_account = accounts[4]

    # Pattern ID
    pattern_id = f"LAYER_{pattern_num:03d}"

    # Start time
    start_time = (
        pd.Timestamp("2026-01-03")
        + pd.Timedelta(
            minutes=random.randint(0, 4 * 24 * 60)
        )
    )

    # Amount
    amount = round(
        random.uniform(50000, 200000),
        2
    )

    chain = [
        (source, intermediary1),
        (intermediary1, intermediary2),
        (intermediary2, intermediary3),
        (intermediary3, final_account)
    ]

    current_time = start_time

    for step, (sender, receiver) in enumerate(chain):

        # 2–8 minute gap between transfers
        if step > 0:
            current_time += pd.Timedelta(
                minutes=random.randint(2, 8)
            )

        new_transactions.append({

            "transaction_id":
                generate_fraud_id(),

            "sender":
                sender,

            "receiver":
                receiver,

            "receiver_name":
                receiver,

            "is_merchant":
                False,

            "transaction_type":
                "Transfer",

            "amount":
                amount,

            "timestamp":
                current_time,

            "atm_id":
                None,

            "city":
                "Delhi",

            "state":
                "Delhi",

            "latitude":
                28.6139,

            "longitude":
                77.2090,

            "fraud_type":
                "Layering",

            "pattern_id":
                pattern_id
        })


# --------------------------------------------------
# 5. Append fraud transactions
# --------------------------------------------------

layering_df = pd.DataFrame(new_transactions)

fraud_transactions = pd.concat(
    [
        fraud_transactions,
        layering_df
    ],
    ignore_index=True
)


# --------------------------------------------------
# 6. Validation
# --------------------------------------------------

print("\nLayering injection completed!")

print(
    "Dataset shape:",
    fraud_transactions.shape
)

print("\nFraud distribution:")
print(
    fraud_transactions["fraud_type"]
    .value_counts()
)

print("\nLayering pattern count:")
print(
    layering_df["pattern_id"].nunique()
)

print("\nTransactions per pattern:")
print(
    layering_df["pattern_id"]
    .value_counts()
    .value_counts()
)

Available customer accounts: 5000

Layering injection completed!
Dataset shape: (148744, 15)

Fraud distribution:
fraud_type
Normal      148544
Layering       200
Name: count, dtype: int64

Layering pattern count:
50

Transactions per pattern:
count
4    50
Name: count, dtype: int64


In [ ]:

pattern_id = "LAYER_001"

chain = (
    fraud_transactions[
        fraud_transactions["pattern_id"] == pattern_id
    ]
    .sort_values("timestamp")
)

print("Pattern:", pattern_id)
print()

for _, row in chain.iterrows():

    print(
        f"{row['timestamp']} | "
        f"{row['sender']} → {row['receiver']} | "
        f"₹{row['amount']:,.2f}"
    )

Pattern: LAYER_001

2026-01-05 06:01:00 | CUST04210 → CUST04529 | ₹124,587.53
2026-01-05 06:07:00 | CUST04529 → CUST01302 | ₹124,587.53
2026-01-05 06:13:00 | CUST01302 → CUST00287 | ₹124,587.53
2026-01-05 06:15:00 | CUST00287 → CUST02740 | ₹124,587.53


In [ ]:
import numpy as np

import random
import pandas as pd

new_smurf_transactions = []

NUM_SMURF_PATTERNS = 50
TRANSACTIONS_PER_PATTERN = 5

# Get customer accounts
customer_ids = sorted(
    set(fraud_transactions["sender"].astype(str))
    |
    set(fraud_transactions["receiver"].astype(str))
)

customer_ids = [
    x for x in customer_ids
    if x.startswith("CUST")
]

for pattern_num in range(1, NUM_SMURF_PATTERNS + 1):

    # 1 source + 5 receivers
    accounts = random.sample(
        customer_ids,
        TRANSACTIONS_PER_PATTERN + 1
    )

    source = accounts[0]
    receivers = accounts[1:]

    pattern_id = f"SMURF_{pattern_num:03d}"

    # Starting time
    current_time = (
        pd.Timestamp("2026-01-03")
        + pd.Timedelta(
            minutes=random.randint(0, 4 * 24 * 60)
        )
    )

    # Total amount distributed
    total_amount = round(
        random.uniform(100000, 300000),
        2
    )

    # Split total amount among receivers
    weights = np.random.dirichlet(
        np.ones(TRANSACTIONS_PER_PATTERN)
    )

    amounts = [
        round(total_amount * weight, 2)
        for weight in weights
    ]

    # Correct rounding difference
    amounts[-1] = round(
        total_amount - sum(amounts[:-1]),
        2
    )

    for i, receiver in enumerate(receivers):

        if i > 0:
            current_time += pd.Timedelta(
                minutes=random.randint(2, 10)
            )

        new_smurf_transactions.append({

            "transaction_id":
                generate_fraud_id(),

            "sender":
                source,

            "receiver":
                receiver,

            "receiver_name":
                receiver,

            "is_merchant":
                False,

            "transaction_type":
                "Transfer",

            "amount":
                amounts[i],

            "timestamp":
                current_time,

            "atm_id":
                None,

            "city":
                "Delhi",

            "state":
                "Delhi",

            "latitude":
                28.6139,

            "longitude":
                77.2090,

            "fraud_type":
                "Smurfing",

            "pattern_id":
                pattern_id
        })


# Convert to DataFrame
smurfing_df = pd.DataFrame(
    new_smurf_transactions
)

# Add to main dataset
fraud_transactions = pd.concat(
    [
        fraud_transactions,
        smurfing_df
    ],
    ignore_index=True
)

print("Smurfing injection completed!")

print("\nDataset shape:")
print(fraud_transactions.shape)

print("\nFraud distribution:")
print(
    fraud_transactions["fraud_type"].value_counts()
)

print("\nSmurfing patterns:")
print(
    smurfing_df["pattern_id"].nunique()
)

print("\nTransactions per Smurfing pattern:")
print(
    smurfing_df["pattern_id"]
    .value_counts()
    .value_counts()
)

Smurfing injection completed!

Dataset shape:
(148994, 15)

Fraud distribution:
fraud_type
Normal      148544
Smurfing       250
Layering       200
Name: count, dtype: int64

Smurfing patterns:
50

Transactions per Smurfing pattern:
count
5    50
Name: count, dtype: int64


In [ ]:

pattern_id = "SMURF_001"

pattern = (
    fraud_transactions[
        fraud_transactions["pattern_id"] == pattern_id
    ]
    .sort_values("timestamp")
)

print("Pattern:", pattern_id)
print()

for _, row in pattern.iterrows():

    print(
        f"{row['timestamp']} | "
        f"{row['sender']} → {row['receiver']} | "
        f"₹{row['amount']:,.2f}"
    )

Pattern: SMURF_001

2026-01-06 23:15:00 | CUST04491 → CUST01969 | ₹49,594.14
2026-01-06 23:20:00 | CUST04491 → CUST00936 | ₹90,423.21
2026-01-06 23:30:00 | CUST04491 → CUST01235 | ₹45,221.50
2026-01-06 23:37:00 | CUST04491 → CUST03353 | ₹12,808.26
2026-01-06 23:41:00 | CUST04491 → CUST02982 | ₹80,124.09


In [ ]:
fraud_transactions.to_csv(
    "/content/drive/MyDrive/AML_Project/dataset/fraud_transactions_v1.csv",
    index=False
)

print("Saved successfully!")

Saved successfully!


In [ ]:
# Cell 14 — Inject Structuring Patterns

import random
import pandas as pd
import numpy as np

new_structuring_transactions = []

NUM_STRUCTURING_PATTERNS = 50
TRANSACTIONS_PER_PATTERN = 6

customer_ids = sorted(
    set(fraud_transactions["sender"].astype(str))
    |
    set(fraud_transactions["receiver"].astype(str))
)

customer_ids = [
    x for x in customer_ids
    if x.startswith("CUST")
]

for pattern_num in range(1, NUM_STRUCTURING_PATTERNS + 1):

    # Select sender and receiver
    sender, receiver = random.sample(customer_ids, 2)

    pattern_id = f"STRUCT_{pattern_num:03d}"

    # Starting time
    current_time = (
        pd.Timestamp("2026-01-03")
        + pd.Timedelta(
            minutes=random.randint(0, 4 * 24 * 60)
        )
    )

    # Base amount for fragmented transactions
    base_amount = random.uniform(40000, 55000)

    for i in range(TRANSACTIONS_PER_PATTERN):

        if i > 0:
            current_time += pd.Timedelta(
                minutes=random.randint(3, 12)
            )

        # Slight variation around the base amount
        amount = round(
            base_amount * random.uniform(0.90, 1.10),
            2
        )

        new_structuring_transactions.append({

            "transaction_id":
                generate_fraud_id(),

            "sender":
                sender,

            "receiver":
                receiver,

            "receiver_name":
                receiver,

            "is_merchant":
                False,

            "transaction_type":
                "Transfer",

            "amount":
                amount,

            "timestamp":
                current_time,

            "atm_id":
                None,

            "city":
                "Delhi",

            "state":
                "Delhi",

            "latitude":
                28.6139,

            "longitude":
                77.2090,

            "fraud_type":
                "Structuring",

            "pattern_id":
                pattern_id
        })


# Convert to DataFrame
structuring_df = pd.DataFrame(
    new_structuring_transactions
)

# Append to main dataset
fraud_transactions = pd.concat(
    [
        fraud_transactions,
        structuring_df
    ],
    ignore_index=True
)

print("Structuring injection completed!")

print("\nDataset shape:")
print(fraud_transactions.shape)

print("\nFraud distribution:")
print(
    fraud_transactions["fraud_type"].value_counts()
)

print("\nStructuring patterns:")
print(
    structuring_df["pattern_id"].nunique()
)

print("\nTransactions per structuring pattern:")
print(
    structuring_df["pattern_id"]
    .value_counts()
    .value_counts()
)

Structuring injection completed!

Dataset shape:
(149294, 15)

Fraud distribution:
fraud_type
Normal         148544
Structuring       300
Smurfing          250
Layering          200
Name: count, dtype: int64

Structuring patterns:
50

Transactions per structuring pattern:
count
6    50
Name: count, dtype: int64


In [ ]:
# Cell 15 — Verify Structuring Pattern

pattern_id = "STRUCT_001"

pattern = (
    fraud_transactions[
        fraud_transactions["pattern_id"] == pattern_id
    ]
    .sort_values("timestamp")
)

print("Pattern:", pattern_id)
print()

for _, row in pattern.iterrows():

    print(
        f"{row['timestamp']} | "
        f"{row['sender']} → {row['receiver']} | "
        f"₹{row['amount']:,.2f}"
    )

print("\nNumber of transactions:", len(pattern))
print("Total amount:", f"₹{pattern['amount'].sum():,.2f}")

Pattern: STRUCT_001

2026-01-04 15:29:00 | CUST03029 → CUST04858 | ₹41,282.61
2026-01-04 15:39:00 | CUST03029 → CUST04858 | ₹37,302.82
2026-01-04 15:50:00 | CUST03029 → CUST04858 | ₹41,853.13
2026-01-04 15:53:00 | CUST03029 → CUST04858 | ₹43,889.45
2026-01-04 15:59:00 | CUST03029 → CUST04858 | ₹43,927.02
2026-01-04 16:05:00 | CUST03029 → CUST04858 | ₹43,681.61

Number of transactions: 6
Total amount: ₹251,936.64


In [ ]:
# Cell 16 — Inject Circular Transaction Patterns

import random
import pandas as pd

new_circular_transactions = []

NUM_CIRCULAR_PATTERNS = 50
TRANSACTIONS_PER_PATTERN = 4

customer_ids = sorted(
    set(fraud_transactions["sender"].astype(str))
    |
    set(fraud_transactions["receiver"].astype(str))
)

customer_ids = [
    x for x in customer_ids
    if x.startswith("CUST")
]

for pattern_num in range(1, NUM_CIRCULAR_PATTERNS + 1):

    # Four accounts create a closed loop
    accounts = random.sample(customer_ids, 4)

    A = accounts[0]
    B = accounts[1]
    C = accounts[2]
    D = accounts[3]

    pattern_id = f"CIRC_{pattern_num:03d}"

    current_time = (
        pd.Timestamp("2026-01-04")
        + pd.Timedelta(
            minutes=random.randint(0, 3 * 24 * 60)
        )
    )

    amount = round(
        random.uniform(50000, 150000),
        2
    )

    chain = [
        (A, B),
        (B, C),
        (C, D),
        (D, A)
    ]

    for step, (sender, receiver) in enumerate(chain):

        if step > 0:
            current_time += pd.Timedelta(
                minutes=random.randint(3, 10)
            )

        new_circular_transactions.append({

            "transaction_id":
                generate_fraud_id(),

            "sender":
                sender,

            "receiver":
                receiver,

            "receiver_name":
                receiver,

            "is_merchant":
                False,

            "transaction_type":
                "Transfer",

            "amount":
                amount,

            "timestamp":
                current_time,

            "atm_id":
                None,

            "city":
                "Delhi",

            "state":
                "Delhi",

            "latitude":
                28.6139,

            "longitude":
                77.2090,

            "fraud_type":
                "Circular",

            "pattern_id":
                pattern_id
        })


# Convert to DataFrame
circular_df = pd.DataFrame(
    new_circular_transactions
)

# Append to main dataset
fraud_transactions = pd.concat(
    [
        fraud_transactions,
        circular_df
    ],
    ignore_index=True
)

print("Circular transaction injection completed!")

print("\nDataset shape:")
print(fraud_transactions.shape)

print("\nFraud distribution:")
print(
    fraud_transactions["fraud_type"].value_counts()
)

print("\nCircular patterns:")
print(
    circular_df["pattern_id"].nunique()
)

print("\nTransactions per circular pattern:")
print(
    circular_df["pattern_id"]
    .value_counts()
    .value_counts()
)

Circular transaction injection completed!

Dataset shape:
(149494, 15)

Fraud distribution:
fraud_type
Normal         148544
Structuring       300
Smurfing          250
Layering          200
Circular          200
Name: count, dtype: int64

Circular patterns:
50

Transactions per circular pattern:
count
4    50
Name: count, dtype: int64


In [ ]:
# Cell 17 — Verify Circular Pattern

pattern_id = "CIRC_001"

pattern = (
    fraud_transactions[
        fraud_transactions["pattern_id"] == pattern_id
    ]
    .sort_values("timestamp")
)

print("Pattern:", pattern_id)
print()

for _, row in pattern.iterrows():

    print(
        f"{row['timestamp']} | "
        f"{row['sender']} → {row['receiver']} | "
        f"₹{row['amount']:,.2f}"
    )

print("\nNumber of transactions:", len(pattern))

Pattern: CIRC_001

2026-01-05 18:54:00 | CUST00966 → CUST00539 | ₹130,827.13
2026-01-05 19:03:00 | CUST00539 → CUST01884 | ₹130,827.13
2026-01-05 19:10:00 | CUST01884 → CUST01237 | ₹130,827.13
2026-01-05 19:13:00 | CUST01237 → CUST00966 | ₹130,827.13

Number of transactions: 4


In [ ]:
# Cell 18 — Inject Mule Account Patterns

import random
import pandas as pd

new_mule_transactions = []

NUM_MULE_PATTERNS = 50

customer_ids = sorted(
    set(fraud_transactions["sender"].astype(str))
    |
    set(fraud_transactions["receiver"].astype(str))
)

customer_ids = [
    x for x in customer_ids
    if x.startswith("CUST")
]

for pattern_num in range(1, NUM_MULE_PATTERNS + 1):

    # 5 source accounts + 1 mule + 1 final account
    accounts = random.sample(customer_ids, 7)

    sources = accounts[:5]
    mule = accounts[5]
    final_account = accounts[6]

    pattern_id = f"MULE_{pattern_num:03d}"

    current_time = (
        pd.Timestamp("2026-01-05")
        + pd.Timedelta(
            minutes=random.randint(0, 2 * 24 * 60)
        )
    )

    # ------------------------------------------
    # Incoming transactions
    # ------------------------------------------

    incoming_amounts = [
        round(random.uniform(20000, 80000), 2)
        for _ in range(5)
    ]

    for i, (source, amount) in enumerate(
        zip(sources, incoming_amounts)
    ):

        if i > 0:
            current_time += pd.Timedelta(
                minutes=random.randint(2, 8)
            )

        new_mule_transactions.append({

            "transaction_id":
                generate_fraud_id(),

            "sender":
                source,

            "receiver":
                mule,

            "receiver_name":
                mule,

            "is_merchant":
                False,

            "transaction_type":
                "Transfer",

            "amount":
                amount,

            "timestamp":
                current_time,

            "atm_id":
                None,

            "city":
                "Delhi",

            "state":
                "Delhi",

            "latitude":
                28.6139,

            "longitude":
                77.2090,

            "fraud_type":
                "Mule",

            "pattern_id":
                pattern_id
        })


    # ------------------------------------------
    # Outgoing transaction from mule
    # ------------------------------------------

    current_time += pd.Timedelta(
        minutes=random.randint(5, 15)
    )

    outgoing_amount = round(
        sum(incoming_amounts) * random.uniform(0.85, 0.98),
        2
    )

    new_mule_transactions.append({

        "transaction_id":
            generate_fraud_id(),

        "sender":
            mule,

        "receiver":
            final_account,

        "receiver_name":
            final_account,

        "is_merchant":
            False,

        "transaction_type":
            "Transfer",

        "amount":
            outgoing_amount,

        "timestamp":
            current_time,

        "atm_id":
            None,

        "city":
            "Delhi",

        "state":
            "Delhi",

        "latitude":
            28.6139,

        "longitude":
            77.2090,

        "fraud_type":
            "Mule",

        "pattern_id":
            pattern_id
    })


# Convert to DataFrame
mule_df = pd.DataFrame(
    new_mule_transactions
)

# Append
fraud_transactions = pd.concat(
    [
        fraud_transactions,
        mule_df
    ],
    ignore_index=True
)

print("Mule account injection completed!")

print("\nDataset shape:")
print(fraud_transactions.shape)

print("\nFraud distribution:")
print(
    fraud_transactions["fraud_type"].value_counts()
)

print("\nMule patterns:")
print(
    mule_df["pattern_id"].nunique()
)

print("\nTransactions per Mule pattern:")
print(
    mule_df["pattern_id"]
    .value_counts()
    .value_counts()
)

Mule account injection completed!

Dataset shape:
(149794, 15)

Fraud distribution:
fraud_type
Normal         148544
Structuring       300
Mule              300
Smurfing          250
Layering          200
Circular          200
Name: count, dtype: int64

Mule patterns:
50

Transactions per Mule pattern:
count
6    50
Name: count, dtype: int64


In [ ]:
# Cell 19 — Verify Mule Pattern

pattern_id = "MULE_001"

pattern = (
    fraud_transactions[
        fraud_transactions["pattern_id"] == pattern_id
    ]
    .sort_values("timestamp")
)

print("Pattern:", pattern_id)
print()

for _, row in pattern.iterrows():

    print(
        f"{row['timestamp']} | "
        f"{row['sender']} → {row['receiver']} | "
        f"₹{row['amount']:,.2f}"
    )

print("\nNumber of transactions:", len(pattern))

mule_account = pattern["receiver"].value_counts().idxmax()

print("Potential mule account:", mule_account)

Pattern: MULE_001

2026-01-06 02:13:00 | CUST01548 → CUST02322 | ₹42,260.14
2026-01-06 02:21:00 | CUST00377 → CUST02322 | ₹36,999.60
2026-01-06 02:26:00 | CUST01568 → CUST02322 | ₹22,784.86
2026-01-06 02:31:00 | CUST04109 → CUST02322 | ₹72,548.21
2026-01-06 02:38:00 | CUST04262 → CUST02322 | ₹52,808.52
2026-01-06 02:51:00 | CUST02322 → CUST04810 | ₹218,080.60

Number of transactions: 6
Potential mule account: CUST02322


In [ ]:
new_rapid_geo_transactions = []

NUM_RAPID_PATTERNS = 50
TRANSACTIONS_PER_PATTERN = 4

# Geographic locations
locations = [
    {
        "city": "New Delhi",
        "state": "Delhi",
        "latitude": 28.6139,
        "longitude": 77.2090
    },
    {
        "city": "Jaipur",
        "state": "Rajasthan",
        "latitude": 26.9124,
        "longitude": 75.7873
    },
    {
        "city": "Mumbai",
        "state": "Maharashtra",
        "latitude": 19.0760,
        "longitude": 72.8777
    },
    {
        "city": "Hyderabad",
        "state": "Telangana",
        "latitude": 17.3850,
        "longitude": 78.4867
    },
    {
        "city": "Bengaluru",
        "state": "Karnataka",
        "latitude": 12.9716,
        "longitude": 77.5946
    },
    {
        "city": "Kolkata",
        "state": "West Bengal",
        "latitude": 22.5726,
        "longitude": 88.3639
    },
    {
        "city": "Chennai",
        "state": "Tamil Nadu",
        "latitude": 13.0827,
        "longitude": 80.2707
    },
    {
        "city": "Ahmedabad",
        "state": "Gujarat",
        "latitude": 23.0225,
        "longitude": 72.5714
    }
]

customer_ids = sorted(
    set(fraud_transactions["sender"].astype(str))
    |
    set(fraud_transactions["receiver"].astype(str))
)

customer_ids = [
    x for x in customer_ids
    if x.startswith("CUST")
]


for pattern_num in range(1, NUM_RAPID_PATTERNS + 1):

    # One account performs transactions
    customer = random.choice(customer_ids)

    # Select 4 geographically distant locations
    selected_locations = random.sample(
        locations,
        TRANSACTIONS_PER_PATTERN
    )

    pattern_id = f"RAPIDGEO_{pattern_num:03d}"

    current_time = (
        pd.Timestamp("2026-01-06")
        + pd.Timedelta(
            minutes=random.randint(0, 24 * 60)
        )
    )

    for i, location in enumerate(selected_locations):

        if i > 0:

            # Very short time gap despite large geographic movement
            current_time += pd.Timedelta(
                minutes=random.randint(3, 8)
            )

        amount = round(
            random.uniform(20000, 150000),
            2
        )

        new_rapid_geo_transactions.append({

            "transaction_id":
                generate_fraud_id(),

            "sender":
                customer,

            "receiver":
                random.choice(customer_ids),

            "receiver_name":
                random.choice(customer_ids),

            "is_merchant":
                False,

            "transaction_type":
                "Transfer",

            "amount":
                amount,

            "timestamp":
                current_time,

            "atm_id":
                None,

            "city":
                location["city"],

            "state":
                location["state"],

            "latitude":
                location["latitude"],

            "longitude":
                location["longitude"],

            "fraud_type":
                "RapidGeographicMovement",

            "pattern_id":
                pattern_id
        })


# Convert to DataFrame
rapid_geo_df = pd.DataFrame(
    new_rapid_geo_transactions
)

# Append to dataset
fraud_transactions = pd.concat(
    [
        fraud_transactions,
        rapid_geo_df
    ],
    ignore_index=True
)

print("Rapid geographic movement injection completed!")

print("\nDataset shape:")
print(fraud_transactions.shape)

print("\nFraud distribution:")
print(
    fraud_transactions["fraud_type"].value_counts()
)

print("\nRapid geographic patterns:")
print(
    rapid_geo_df["pattern_id"].nunique()
)

print("\nTransactions per rapid geographic pattern:")
print(
    rapid_geo_df["pattern_id"]
    .value_counts()
    .value_counts()
)

Rapid geographic movement injection completed!

Dataset shape:
(149994, 15)

Fraud distribution:
fraud_type
Normal                     148544
Mule                          300
Structuring                   300
Smurfing                      250
Layering                      200
Circular                      200
RapidGeographicMovement       200
Name: count, dtype: int64

Rapid geographic patterns:
50

Transactions per rapid geographic pattern:
count
4    50
Name: count, dtype: int64


In [ ]:
pattern_id = "RAPIDGEO_001"

pattern = (
    fraud_transactions[
        fraud_transactions["pattern_id"] == pattern_id
    ]
    .sort_values("timestamp")
)

print("Pattern:", pattern_id)
print()

for _, row in pattern.iterrows():

    print(
        f"{row['timestamp']} | "
        f"{row['city']} | "
        f"{row['latitude']:.4f}, {row['longitude']:.4f} | "
        f"{row['sender']} → {row['receiver']} | "
        f"₹{row['amount']:,.2f}"
    )

print("\nNumber of transactions:", len(pattern))

Pattern: RAPIDGEO_001

2026-01-06 02:05:00 | Kolkata | 22.5726, 88.3639 | CUST01413 → CUST03863 | ₹103,975.63
2026-01-06 02:09:00 | New Delhi | 28.6139, 77.2090 | CUST01413 → CUST04107 | ₹90,466.48
2026-01-06 02:12:00 | Hyderabad | 17.3850, 78.4867 | CUST01413 → CUST01469 | ₹48,666.64
2026-01-06 02:16:00 | Chennai | 13.0827, 80.2707 | CUST01413 → CUST00749 | ₹51,180.05

Number of transactions: 4


In [ ]:
fraud_transactions.to_csv(
    "/content/drive/MyDrive/AML_Project/dataset/fraud_transactions_complete.csv",
    index=False
)

print("Final fraud-injected dataset saved!")
print(fraud_transactions.shape)

Final fraud-injected dataset saved!
(149994, 15)
